# Langkah 4 — Fuzzy Sugeno
Notebook ini mengambil output probabilitas dari ANN (`y_prob_test.npy`) dan memproses
nya melalui sistem inferensi **Fuzzy Sugeno** untuk menghasilkan keputusan akhir deteksi fraud.

### Alur:
1. Load probabilitas ANN
2. Definisikan membership function (MF)
3. Hitung derajat keanggotaan (fuzzifikasi)
4. Terapkan rule base
5. Defuzzifikasi (Weighted Average — metode Sugeno)
6. Evaluasi hasil akhir
7. Simpan model fuzzy & hasil prediksi

## 0. Import Library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay
)
import joblib
import os

# Buat folder output jika belum ada
os.makedirs('../backend/model', exist_ok=True)
os.makedirs('processed', exist_ok=True)

print('Library berhasil diimport!')

## 1. Load Data

In [ ]:
# Load probabilitas output ANN dan label asli
y_prob_test = np.load('processed/y_prob_test.npy')
y_test      = np.load('processed/y_test.npy')

print(f'Jumlah data test  : {len(y_prob_test)}')
print(f'Min prob          : {y_prob_test.min():.4f}')
print(f'Max prob          : {y_prob_test.max():.4f}')
print(f'Mean prob         : {y_prob_test.mean():.4f}')
print(f'Fraud (label=1)   : {y_test.sum().astype(int)}')
print(f'Legit (label=0)   : {(y_test == 0).sum()}')

## 2. Definisi Membership Function (MF)

Variabel input: **prob_fraud** (probabilitas output ANN, range 0–1)

Tiga himpunan fuzzy:
- **Low** → kemungkinan bukan fraud
- **Medium** → ambigu / perlu investigasi
- **High** → kemungkinan fraud tinggi

Variabel output (Sugeno — konstanta):
- Low → 0.0
- Medium → 0.5
- High → 1.0

In [ ]:
# Universe of discourse untuk input
x = np.linspace(0, 1, 1000)

# ---- Membership Functions (trapesium & segitiga) ----
# Low  : [0, 0, 0.3, 0.5]
# Medium: [0.3, 0.5, 0.7]
# High : [0.5, 0.7, 1.0, 1.0]

mf_low    = fuzz.trapmf(x, [0.0, 0.0, 0.30, 0.50])
mf_medium = fuzz.trimf (x, [0.30, 0.50, 0.70])
mf_high   = fuzz.trapmf(x, [0.50, 0.70, 1.00, 1.00])

# ---- Plot MF ----
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, mf_low,    'b-',  linewidth=2, label='Low (Legit)')
ax.plot(x, mf_medium, 'y-',  linewidth=2, label='Medium (Ambiguous)')
ax.plot(x, mf_high,   'r-',  linewidth=2, label='High (Fraud)')
ax.set_xlabel('Probabilitas Fraud (output ANN)', fontsize=12)
ax.set_ylabel('Derajat Keanggotaan', fontsize=12)
ax.set_title('Membership Function — Fuzzy Sugeno WASPADA', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fuzzy_membership_function.png', dpi=150)
plt.show()
print('MF berhasil didefinisikan dan disimpan.')

## 3. Fuzzifikasi
Hitung derajat keanggotaan setiap sampel terhadap ketiga himpunan fuzzy.

In [ ]:
def fuzzify(prob):
    """Hitung derajat keanggotaan untuk satu nilai probabilitas."""
    mu_low    = fuzz.interp_membership(x, mf_low,    prob)
    mu_medium = fuzz.interp_membership(x, mf_medium, prob)
    mu_high   = fuzz.interp_membership(x, mf_high,   prob)
    return mu_low, mu_medium, mu_high

# Fuzzifikasi seluruh dataset test
mu_low_arr    = np.array([fuzz.interp_membership(x, mf_low,    p) for p in y_prob_test])
mu_medium_arr = np.array([fuzz.interp_membership(x, mf_medium, p) for p in y_prob_test])
mu_high_arr   = np.array([fuzz.interp_membership(x, mf_high,   p) for p in y_prob_test])

print('Fuzzifikasi selesai.')
print(f'Contoh (sampel pertama, prob={y_prob_test[0]:.4f}):')
print(f'  mu_low    = {mu_low_arr[0]:.4f}')
print(f'  mu_medium = {mu_medium_arr[0]:.4f}')
print(f'  mu_high   = {mu_high_arr[0]:.4f}')

## 4. Rule Base + Inferensi Sugeno

Rule (Sugeno — output berupa konstanta):

| Rule | IF prob_fraud IS | THEN output |
|------|------------------|-------------|
| R1   | Low              | 0.0         |
| R2   | Medium           | 0.5         |
| R3   | High             | 1.0         |

**Defuzzifikasi Sugeno (Weighted Average):**

$$z^* = \frac{\mu_{low} \cdot 0 + \mu_{medium} \cdot 0.5 + \mu_{high} \cdot 1.0}{\mu_{low} + \mu_{medium} + \mu_{high}}$$

In [ ]:
# Konstanta output Sugeno
C_LOW    = 0.0
C_MEDIUM = 0.5
C_HIGH   = 1.0

def sugeno_defuzz(mu_low, mu_medium, mu_high):
    """Weighted average defuzzification (Sugeno)."""
    numerator   = mu_low * C_LOW + mu_medium * C_MEDIUM + mu_high * C_HIGH
    denominator = mu_low + mu_medium + mu_high
    if denominator == 0:
        return 0.0
    return numerator / denominator

# Hitung skor fuzzy untuk semua sampel
fuzzy_score = np.array([
    sugeno_defuzz(mu_low_arr[i], mu_medium_arr[i], mu_high_arr[i])
    for i in range(len(y_prob_test))
])

print('Defuzzifikasi Sugeno selesai.')
print(f'Contoh 5 skor fuzzy pertama: {fuzzy_score[:5].round(4)}')

## 5. Threshold & Prediksi Akhir
Konversi skor fuzzy ke label biner menggunakan threshold 0.5.

In [ ]:
THRESHOLD = 0.5
y_pred_fuzzy = (fuzzy_score >= THRESHOLD).astype(int)

print(f'Threshold : {THRESHOLD}')
print(f'Prediksi Fraud  : {y_pred_fuzzy.sum()}')
print(f'Prediksi Legit  : {(y_pred_fuzzy == 0).sum()}')

## 6. Evaluasi Performa

In [ ]:
print('='*55)
print('   EVALUASI — ANN + FUZZY SUGENO (WASPADA)')
print('='*55)
print(classification_report(y_test, y_pred_fuzzy,
                             target_names=['Legit', 'Fraud']))

auc = roc_auc_score(y_test, fuzzy_score)
print(f'ROC-AUC Score : {auc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Confusion Matrix ----
cm = confusion_matrix(y_test, y_pred_fuzzy)
disp = ConfusionMatrixDisplay(cm, display_labels=['Legit', 'Fraud'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — ANN + Fuzzy Sugeno', fontsize=13)

# ---- ROC Curve ----
fpr, tpr, _ = roc_curve(y_test, fuzzy_score)
axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — ANN + Fuzzy Sugeno', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fuzzy_evaluation.png', dpi=150)
plt.show()
print('Grafik evaluasi disimpan.')

In [ ]:
# ---- Distribusi Skor Fuzzy ----
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(fuzzy_score[y_test == 0], bins=80, alpha=0.6, color='steelblue', label='Legit')
ax.hist(fuzzy_score[y_test == 1], bins=80, alpha=0.7, color='crimson',   label='Fraud')
ax.axvline(THRESHOLD, color='black', linestyle='--', linewidth=1.5, label=f'Threshold={THRESHOLD}')
ax.set_xlabel('Skor Fuzzy Sugeno')
ax.set_ylabel('Frekuensi')
ax.set_title('Distribusi Skor Fuzzy — WASPADA', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fuzzy_score_distribution.png', dpi=150)
plt.show()

## 7. Simpan Model & Hasil

In [ ]:
# Simpan parameter fuzzy sebagai dict (dipakai oleh API)
fuzzy_params = {
    'mf_low'    : {'type': 'trapmf', 'params': [0.0, 0.0, 0.30, 0.50]},
    'mf_medium' : {'type': 'trimf',  'params': [0.30, 0.50, 0.70]},
    'mf_high'   : {'type': 'trapmf', 'params': [0.50, 0.70, 1.00, 1.00]},
    'constants' : {'low': C_LOW, 'medium': C_MEDIUM, 'high': C_HIGH},
    'threshold' : THRESHOLD
}

joblib.dump(fuzzy_params, '../backend/model/fuzzy_params.pkl')
print('fuzzy_params.pkl disimpan di backend/model/')

# Simpan skor fuzzy untuk analisis lanjut
np.save('processed/y_fuzzy_score.npy', fuzzy_score)
np.save('processed/y_pred_fuzzy.npy',  y_pred_fuzzy)
print('y_fuzzy_score.npy & y_pred_fuzzy.npy disimpan di notebook/processed/')

In [ ]:
# Ringkasan final
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print('\n' + '='*55)
print('   RINGKASAN PERFORMA — ANN + FUZZY SUGENO')
print('='*55)
print(f'Accuracy  : {accuracy_score(y_test, y_pred_fuzzy):.4f}')
print(f'Precision : {precision_score(y_test, y_pred_fuzzy):.4f}')
print(f'Recall    : {recall_score(y_test, y_pred_fuzzy):.4f}')
print(f'F1-Score  : {f1_score(y_test, y_pred_fuzzy):.4f}')
print(f'ROC-AUC   : {auc:.4f}')
print('='*55)
print()
print('File yang tersimpan:')
print('  backend/model/fuzzy_params.pkl')
print('  notebook/processed/y_fuzzy_score.npy')
print('  notebook/processed/y_pred_fuzzy.npy')
print('  notebook/fuzzy_membership_function.png')
print('  notebook/fuzzy_evaluation.png')
print('  notebook/fuzzy_score_distribution.png')
print()
print('Langkah 4 selesai! Lanjut ke Langkah 5 — Backend API (app.py).')